# Bias-Aware Resume Matching using SBERT

In this project, I am building the main resume-job matching pipeline for my project. The goal is to compare resumes with job descriptions and generate a matching score for each pair.

For the main quantitative part, I am using Sentence-BERT because it gives stable numerical embeddings for both resumes and job descriptions. After converting the text into embeddings, I calculate cosine similarity between them. A higher score means the resume is more closely related to the job description.

These scores will be used later for the fairness audit. I will compare the scores of original resumes with counterfactual resume versions where only demographic signals such as name, pronouns, or university are changed.

Since the project feedback also suggested examining a current LLM, I will include a separate LLM-based demo later. The LLM part will be used to compare how a generative model explains or rates resume-job fit, while SBERT will remain the main model for the numerical fairness analysis.

In [1]:
# Install the required libraries for this notebook
!pip install sentence-transformers pandas scikit-learn

In [2]:
# Import libraries for data handling, text embeddings, and similarity calculation
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Create folders for input data and output results
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [3]:
# Upload the dataset files from your computer
from google.colab import files

uploaded = files.upload()

# Move uploaded files into the data folder
for filename in uploaded.keys():
    os.rename(filename, f"data/{filename}")

print("Files uploaded and moved to the data folder.")

Saving jobs.csv to jobs.csv
Saving resume_variants.csv to resume_variants.csv
Files uploaded and moved to the data folder.


In [4]:
# Load the job descriptions and resume variants
jobs = pd.read_csv("data/jobs.csv")
resumes = pd.read_csv("data/resume_variants.csv")

print("Number of jobs:", jobs.shape)
print("Number of resume variants:", resumes.shape)

display(jobs.head())
display(resumes.head())

Number of jobs: (5, 5)
Number of resume variants: (40, 8)


,job_id,domain,title,company_name,job_description
0,J1,Software Engineering,Software Engineer,GOYT,Job Description:GOYT is seeking a skilled and ...
1,J2,Finance,Financial Analyst,Aya Healthcare,Aya Healthcare has an exciting 26-week contrac...
2,J3,Marketing,Marketing Coordinator,Corcoran Sawyer Smith,Job descriptionA leading real estate firm in N...
3,J4,Healthcare,Clinical Data Analyst,Insight Global,Must Have: 2+ years of current experience in a...
4,J5,Education,Academic Advisor,Kennesaw State University,About Us Are you ready to join a community lea...


,resume_id,domain,version,changed_signal,name,pronouns,university,resume_text
0,R1,Software Engineering,original,none,Emily Johnson,she/her,Stanford University,"Emily Johnson, she/her. B.S. in Computer Scien..."
1,R1,Software Engineering,name_changed,name,Aisha Khan,she/her,Stanford University,"Aisha Khan, she/her. B.S. in Computer Science ..."
2,R1,Software Engineering,pronoun_changed,pronoun,Emily Johnson,he/him,Stanford University,"Emily Johnson, he/him. B.S. in Computer Scienc..."
3,R1,Software Engineering,university_changed,university,Emily Johnson,she/her,Morgan State University,"Emily Johnson, she/her. B.S. in Computer Scien..."
4,R2,Software Engineering,original,none,Michael Smith,he/him,Massachusetts Institute of Technology,"Michael Smith, he/him. B.S. in Software Engine..."


In [5]:
# The resume_variants.csv file already has a resume_text column.
# I will use that text directly for SBERT scoring.

# The jobs.csv file has job details split across columns.
# Here I combine the useful job information into one text field.

def create_job_text(row):
    return (
        str(row["title"]) + " " +
        str(row["domain"]) + " " +
        str(row["company_name"]) + " " +
        str(row["job_description"])
    )

jobs["job_text"] = jobs.apply(create_job_text, axis=1)

display(jobs[["job_id", "title", "job_text"]].head())
display(resumes[["resume_id", "version", "changed_signal", "resume_text"]].head())

,job_id,title,job_text
0,J1,Software Engineer,Software Engineer Software Engineering GOYT Jo...
1,J2,Financial Analyst,Financial Analyst Finance Aya Healthcare Aya H...
2,J3,Marketing Coordinator,Marketing Coordinator Marketing Corcoran Sawye...
3,J4,Clinical Data Analyst,Clinical Data Analyst Healthcare Insight Globa...
4,J5,Academic Advisor,Academic Advisor Education Kennesaw State Univ...


,resume_id,version,changed_signal,resume_text
0,R1,original,none,"Emily Johnson, she/her. B.S. in Computer Scien..."
1,R1,name_changed,name,"Aisha Khan, she/her. B.S. in Computer Science ..."
2,R1,pronoun_changed,pronoun,"Emily Johnson, he/him. B.S. in Computer Scienc..."
3,R1,university_changed,university,"Emily Johnson, she/her. B.S. in Computer Scien..."
4,R2,original,none,"Michael Smith, he/him. B.S. in Software Engine..."


In [6]:
# Load the Sentence-BERT model
# This model converts resume and job text into numerical embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

print("SBERT model loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT model loaded successfully.


In [7]:
# Compare every resume variant with every job description
# Cosine similarity is used as the resume-job matching score

results = []

for _, resume_row in resumes.iterrows():
    resume_embedding = model.encode([resume_row["resume_text"]])

    for _, job_row in jobs.iterrows():
        job_embedding = model.encode([job_row["job_text"]])
        score = cosine_similarity(resume_embedding, job_embedding)[0][0]

        results.append({
            "resume_id": resume_row["resume_id"],
            "version": resume_row["version"],
            "changed_signal": resume_row["changed_signal"],
            "job_id": job_row["job_id"],
            "job_title": job_row["title"],
            "similarity_score": score
        })

scores_df = pd.DataFrame(results)

display(scores_df.head())
print("Total resume-job pairs scored:", len(scores_df))

,resume_id,version,changed_signal,job_id,job_title,similarity_score
0,R1,original,none,J1,Software Engineer,0.529075
1,R1,original,none,J2,Financial Analyst,0.170317
2,R1,original,none,J3,Marketing Coordinator,0.319204
3,R1,original,none,J4,Clinical Data Analyst,0.326272
4,R1,original,none,J5,Academic Advisor,0.337290


Total resume-job pairs scored: 200


In [8]:
# Save the SBERT matching scores to the results folder
scores_df.to_csv("results/sbert_scores.csv", index=False)

print("SBERT matching scores saved to results/sbert_scores.csv")

SBERT matching scores saved to results/sbert_scores.csv


In [9]:
# Download the result file so it can be uploaded to GitHub
from google.colab import files

files.download("results/sbert_scores.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>